In [6]:
from dataset import SiameseMelanomaClassifierDataset
from modules import SiameseNetwork
from torch.utils.data import DataLoader
from train import Trainer
import torch
import pandas as pd
import os
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm

device = torch.device('mps' if torch.mps.is_available() else 'cpu')

train_dataset = SiameseMelanomaClassifierDataset(
    'data/cleaned/train_pairs.csv',
    'data/cleaned/train_images',
    mode='train'
)

val_dataset = SiameseMelanomaClassifierDataset(
    'data/cleaned/validation_pairs.csv',
    'data/cleaned/validation_images',
    mode='val'
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

(img_1, img_2), label = next(iter(train_loader))
print(img_1.shape, img_2.shape, label)
# torchvision.transforms.functional.to_pil_image(img_1[0, :, :, :]).show()

# network = SiameseNetwork()
# trainer = Trainer(network, train_loader, val_loader, device, lr=0.0001)
# trainer.train(epochs=5)


torch.Size([8, 3, 224, 224]) torch.Size([8, 3, 224, 224]) tensor([0, 0, 0, 0, 0, 0, 0, 1])


In [ ]:
# Compute train mean
device = torch.device('mps' if torch.mps.is_available() else 'cpu')

train_csv_path = 'data/cleaned/train.csv'
img_dir = 'data/cleaned/train_images'

data = pd.read_csv(train_csv_path)
image_ids = data['image_name'].unique()

means, stds = [], []

for img_id in tqdm(image_ids, desc="train mean+std"):
    path = os.path.join(img_dir, f"{img_id}.jpg")
    img = Image.open(path).convert('RGB')
    tensor = transforms.ToTensor()(img).to(device)
    means.append(tensor.mean(dim=(1, 2)))
    stds.append(tensor.std(dim=(1, 2)))

mean = torch.stack(means).mean(0)
std = torch.stack(stds).mean(0)

print(f"mean: {mean.tolist()}")
print(f"std: {std.tolist()}")

Computing train mean/std:   0%|          | 0/22890 [00:00<?, ?it/s]